# Case-001 Private Intake Priority Gate

This public-safe notebook ranks which private record domains should be indexed first. It does not read raw records, store identifiers, diagnose, or recommend treatment.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Literal

Status = Literal["missing", "indexed", "deidentified", "public_ready"]


@dataclass(frozen=True)
class IntakeDomain:
    """One private record domain needed before public case routing."""

    name: str
    status: Status
    diagnosis_dependency: int
    safety_dependency: int
    referral_dependency: int
    blocker: str

    @property
    def priority_score(self) -> int:
        """Rank missing domains by diagnosis, safety, and referral value."""
        if self.status == "public_ready":
            return 0
        weight = 3 if self.status == "missing" else 1
        return weight * (
            self.diagnosis_dependency
            + self.safety_dependency
            + self.referral_dependency
        )

In [ ]:
domains = [
    IntakeDomain(
        "diagnosis, HPLC, electrophoresis, and genotype records",
        "missing",
        3,
        2,
        3,
        "diagnosis_genotype_private_index_missing",
    ),
    IntakeDomain(
        "transfusion dates, volumes, body weight, and Hb response",
        "missing",
        2,
        3,
        2,
        "transfusion_log_private_index_missing",
    ),
    IntakeDomain(
        "blood-bank, antibody, DAT, and matching records",
        "missing",
        2,
        3,
        2,
        "immune_blood_bank_private_index_missing",
    ),
    IntakeDomain(
        "ferritin, LIC, cardiac T2*, chelation, and organ-risk records",
        "missing",
        2,
        3,
        2,
        "iron_organ_private_index_missing",
    ),
    IntakeDomain(
        "prior advanced-therapy, drug, and trial access review",
        "missing",
        1,
        1,
        3,
        "access_review_private_index_missing",
    ),
]

ranked_domains = sorted(
    domains,
    key=lambda domain: (domain.priority_score, domain.safety_dependency),
    reverse=True,
)
[(domain.name, domain.priority_score, domain.blocker) for domain in ranked_domains]

In [ ]:
def classify_intake(record_domains: list[IntakeDomain]) -> str:
    """Classify whether private records are ready for public-safe extraction."""
    if any(domain.status == "missing" for domain in record_domains):
        return "private_intake_index_needed"
    if any(domain.status == "indexed" for domain in record_domains):
        return "deidentified_timeline_candidate_needed"
    if any(domain.status == "deidentified" for domain in record_domains):
        return "public_summary_candidate_needs_release_check"
    return "public_summary_ready"


result = {
    "case_id": "case-001",
    "current_label": classify_intake(domains),
    "top_blockers": [domain.blocker for domain in ranked_domains[:3]],
    "raw_record_handling": "private_ignored_workspace_only",
    "patient_specific_claims": "blocked",
}
result

## Decision

Durable label: `private_intake_index_needed`. The next practical work is a private ignored source index and de-identified timeline candidate, followed by privacy scanning and public release review. Biomedical claims remain separate from Quranic motivation and patient-specific treatment advice remains blocked.
